# DTW sobre producto×cliente: densificar, escalar por la media y clusterizar

Este notebook reemplaza el enfoque de `02_DTW_clusters` en el punto donde ese se
quedaba corto. Allí la tabla de niveles decía, sobre el par producto-cliente:

| Nivel | Series | Distancias | Matriz | ¿Se puede? |
|---|---|---|---|---|
| par producto×cliente | 374.439 | 1.105 millones | 17,7 GB | **No con este algoritmo** |

El problema no es DTW: es el **clustering jerárquico**, que necesita las
distancias de todos contra todos en memoria a la vez. Acá se cambia el algoritmo,
no el nivel.

## Lo que cambia

**1 — k-means con DTW en vez de jerárquico.** Cada iteración compara las `n` series
contra `k` centroides, no contra las otras `n`. El costo pasa de `O(n²)` a `O(n·k)`:
con 47.000 pares y k=8 son 376.000 distancias por iteración en vez de 1.105
millones, y la memoria pasa de 17,7 GB a unos pocos MB. El centroide se actualiza
con **DBA** (*DTW Barycenter Averaging*): se alinea cada serie del cluster contra el
centroide y se promedian los puntos que quedaron alineados. Promediar punto a punto
sin alinear sería volver a la media euclídea y perdería todo lo que DTW aporta.

**2 — Densificación con ceros desde la primera venta.** Una vez que un par
producto-cliente vendió por primera vez, los meses sin registro **no son huecos: son
ceros**. Ese cliente existió, ese producto existió, y no hubo venta. Dejarlos como
nulos hace que DTW compare series de largos distintos y trate la ausencia como
"no observado", cuando en realidad es la información más fuerte que hay sobre un par
intermitente. Antes de la primera venta sí es ausencia real, y esos meses no entran.

**3 — Escalado por la media.** DTW compara distancias absolutas punto a punto, así
que sin escalar el clustering agrupa por **volumen** y no por forma. Dividir por la
media propia deja cada serie en "veces su propio promedio": adimensional, con la
media en 1, y —a diferencia del z-score— **conserva los ceros como ceros**. En series
tan intermitentes como estas eso importa: con z-score un mes sin venta cae en un
número negativo distinto para cada par, y dos pares igual de intermitentes dejan de
parecerse. Se comparan las alternativas en la grilla, pero la media es la apuesta.

## Lo que hay que tener presente

- **El silhouette se mide sobre una muestra.** Calcularlo sobre los 47.000 pares
  pediría la misma matriz de 17,7 GB que estamos evitando. Se toma una muestra
  aleatoria de series, se calcula la matriz completa de ese subconjunto y se evalúa
  ahí. Es una estimación, y se imprime con cuántas series se hizo.
- **k-means con DTW no es determinista por construcción**, depende de la
  inicialización. Se usa k-means++ con semilla fija, así que la corrida es
  reproducible, pero dos semillas pueden dar particiones distintas.
- **El corte anti-leakage** (`mes_corte`) es el mismo que en `02_DTW_clusters`: si
  estas etiquetas van a ser feature de un modelo, el clustering no puede haber visto
  los meses de validación.

## 0 — Ambiente y paleta

In [ ]:
import gc, json, os, time
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from dtaidistance import dtw
from sklearn.metrics import silhouette_score


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local ultimo."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    local = Path(r"C:\Users\Natalia\labo3-bucket")
    if local.is_dir():
        return local
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
DIR_PRE  = BUCKET / "datasets" / "preprocesado"
DIR_OUT  = BUCKET / "datasets_fe"        # de aca lo lee 02_FE
DIR_RUNS = BUCKET / "exp_clusters_pc"    # una carpeta por configuracion
DIR_OUT.mkdir(parents=True, exist_ok=True)
DIR_RUNS.mkdir(parents=True, exist_ok=True)

SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ   = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRILLA, EJE_C, FONDO = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO,
    "axes.edgecolor": EJE_C, "axes.labelcolor": TINTA2,
    "text.color": TINTA, "xtick.color": MUDO, "ytick.color": MUDO,
    "grid.color": GRILLA, "grid.linewidth": .8,
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "figure.dpi": 110,
    "legend.frameon": False,
})


def limpiar(ax, titulo=None, y=None, x=None):
    if titulo:
        ax.set_title(titulo, color=TINTA, loc="left", pad=10)
    if y:
        ax.set_ylabel(y)
    if x:
        ax.set_xlabel(x)
    ax.grid(axis="x", visible=False)
    return ax


def guardar(fig, nombre, mostrar=True):
    d = globals().get("DIR_RUN")
    if d is None:
        raise RuntimeError("DIR_RUN no definido: corre primero la celda de palancas.")
    p = Path(d) / f"{nombre}.png"
    fig.savefig(p, dpi=140, bbox_inches="tight", facecolor=FONDO)
    print(f"   [fig] {p.relative_to(BUCKET)}")
    plt.show() if mostrar else plt.close(fig)
    return p


# dtaidistance necesita el backend C para que esto sea viable: en Python puro cada
# distancia es ~1000x mas lenta y el notebook no termina nunca.
_C_OK = dtw.try_import_c()
print(f"BUCKET: {BUCKET}")
print(f"DTW con backend C: {_C_OK}")
if not _C_OK:
    print("   ATENCION: sin backend C esto es inusable. pip install -U dtaidistance")

# warping_path no acepta 'window' en todas las versiones de dtaidistance. Se prueba una
# vez y el resto del notebook usa el resultado, en vez de que reviente a mitad del DBA.
try:
    dtw.warping_path(np.zeros(5), np.zeros(5), window=2)
    _WP_WINDOW = True
except TypeError:
    _WP_WINDOW = False
print(f"warping_path acepta window: {_WP_WINDOW}")

## 1 — Palancas

In [ ]:
PARAM = {
    # ── DE DONDE SALEN LAS SERIES ────────────────────────────────────────
    # 'crudo'        -> sell-in.txt.gz + tb_productos. Todo desde cero.
    # 'preprocesado' -> el parquet de 01_Preprocesamiento, que ya viene agrupado por
    #                   producto-cliente-mes y con las categorias pegadas. Mas rapido,
    #                   y respeta los filtros con los que se genero (tgtFilter, smplN).
    'fuente': 'preprocesado',
    # Nombre exacto del parquet en datasets/preprocesado/. None = el mas grande.
    'archivo_preprocesado': None,

    # ── DENSIFICACION ────────────────────────────────────────────────────
    # Que hacer con los meses sin registro de un par producto-cliente:
    #   'desde_nacimiento' -> desde su PRIMERA venta hasta el fin del panel, todo 0.
    #                         Un par que dejo de comprar sigue teniendo ceros: la
    #                         discontinuacion es parte de la forma de la serie.
    #   'vida'             -> solo entre su primera y su ultima venta. El par
    #                         discontinuado termina ahi y su serie es mas corta.
    # 'desde_nacimiento' es el default porque es lo que se pidio y porque hace las
    # series comparables: todas terminan en el mismo mes.
    'densificar': 'desde_nacimiento',

    # ── ANTI-LEAKAGE ─────────────────────────────────────────────────────
    # Solo meses ESTRICTAMENTE ANTERIORES a este corte. Debe coincidir con el ultimo
    # mes de train de 03_Optuna (default alli: 201905 -> corte 201906).
    # None = toda la historia. SOLO para exploracion, nunca para generar features.
    'mes_corte': 201906,

    # Minimo de meses observados (ya densificados) para entrar al clustering.
    # Un par con 3 meses no tiene forma que clusterizar.
    'min_meses': 18,

    # ── ESCALADO ─────────────────────────────────────────────────────────
    # 'media'  -> x / media(x). La serie queda en "veces su propio promedio", con
    #             media 1 y los CEROS SIGUEN SIENDO CERO. Es la apuesta de este
    #             notebook: en series intermitentes el cero es informacion, y hay que
    #             que dos pares igual de intermitentes se parezcan.
    # 'zscore' -> (x - media) / desvio. Forma pura, pero manda los ceros a un negativo
    #             distinto en cada serie.
    # 'maximo' -> x / max(x), a [0,1]. Conserva los ceros y acota, pero un solo mes
    #             excepcional le aplasta toda la serie.
    # 'ninguno'-> sin escalar. Esta para MOSTRAR que sin escalar se agrupa por volumen.
    'escalado': 'media',

    # ── DTW ──────────────────────────────────────────────────────────────
    # Banda de Sakoe-Chiba: cuantos meses de desfase se permiten al alinear.
    # None = sin restriccion (mas lento y permite alineaciones absurdas).
    'window': 3,

    # ── K-MEANS CON DTW ──────────────────────────────────────────────────
    'k': 6,
    'max_iter': 15,
    # Se corta cuando cambia de cluster menos de esta fraccion de las series.
    'tol_cambio': 0.01,
    # Pasos de DBA por iteracion. 1 alcanza: el centroide se refina igual porque la
    # iteracion siguiente lo vuelve a promediar.
    'dba_iters': 1,

    # ── MUESTREO ─────────────────────────────────────────────────────────
    # Cuantos pares entran al ajuste FINAL. None = todos.
    # El cuello de botella es n_pares x k distancias DTW por iteracion.
    'muestra_pares': None,
    # Cuantos pares se usan en la GRILLA de parametros. La grilla ajusta muchas veces,
    # asi que va con menos: sirve para ordenar configuraciones, no para la final.
    'muestra_grilla': 4000,
    # Cuantas series se usan para estimar el silhouette (matriz completa de esta
    # muestra: 3000 -> 9M distancias -> ~70 MB, manejable).
    'muestra_silhouette': 3000,

    # ── GRILLA DE PARAMETROS DEL DTW ─────────────────────────────────────
    'grilla_window':   [1, 3, 6, None],
    'grilla_escalado': ['media', 'zscore', 'maximo'],
    'grilla_k':        [4, 6, 8, 10],

    'semilla': 102191,
}

ESCALADOS = ('media', 'zscore', 'maximo', 'ninguno')
if PARAM['escalado'] not in ESCALADOS:
    raise ValueError(f"escalado invalido: {PARAM['escalado']!r}. Opciones: {ESCALADOS}")
if PARAM['densificar'] not in ('desde_nacimiento', 'vida'):
    raise ValueError(f"densificar invalido: {PARAM['densificar']!r}")
if PARAM['fuente'] not in ('crudo', 'preprocesado'):
    raise ValueError(f"fuente invalida: {PARAM['fuente']!r}")

RNG = np.random.default_rng(PARAM['semilla'])
CATS = ["cat1", "cat2", "cat3", "brand"]

# El nombre de la carpeta arrastra todas las palancas que cambian el resultado, para
# que dos configuraciones no se pisen. k NO va: comparar k es parte de la exploracion
# de esta misma configuracion, y lo que depende de k lleva _k{K} en su propio nombre.
SLUG = (f"pc_{PARAM['densificar']}_{PARAM['escalado']}"
        f"_w{PARAM['window']}_min{PARAM['min_meses']}_corte{PARAM['mes_corte']}")
DIR_RUN = DIR_RUNS / SLUG
DIR_RUN.mkdir(parents=True, exist_ok=True)
with open(DIR_RUN / "config.json", "w", encoding="utf-8") as f:
    json.dump(PARAM, f, indent=2, ensure_ascii=False)

print(f"carpeta : {DIR_RUN.relative_to(BUCKET)}")
print(f"fuente  : {PARAM['fuente']}   densificacion: {PARAM['densificar']}")
print(f"escalado: {PARAM['escalado']}   window: {PARAM['window']}   k: {PARAM['k']}")

## 2 — Panel producto-cliente-mes

Del crudo o del parquet de `01_Preprocesamiento`, según la palanca. En los dos casos
queda lo mismo: una fila por `(product_id, customer_id, periodo)` con `tn`, más las
categorías del producto para caracterizar después.

In [ ]:
t0 = time.time()


def a_m(p):
    """AAAAMM -> indice de mes continuo, para poder sumar y restar meses."""
    return (p // 100) * 12 + (p % 100)


def m_a_periodo(m):
    return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1


def normalizar_periodo(df: pl.DataFrame) -> pl.DataFrame:
    """Deja 'periodo' como Int64 AAAAMM, venga como Date, string o entero.

    01_Preprocesamiento lo graba como Date y el crudo como entero AAAAMM. Sin unificar,
    el // 100 de a_m() revienta con "// not allowed on date and dyn int". Se unifica
    ACA y no dentro de a_m() para que el resto del notebook (mes_corte, m_nace,
    int_ranges) trabaje siempre contra enteros.
    """
    dt = df.schema['periodo']
    try:
        temporal = dt.is_temporal()
    except AttributeError:
        temporal = dt in (pl.Date, pl.Datetime)
    if temporal:
        print(f"  periodo venia como {dt} -> se convierte a Int64 AAAAMM")
        return df.with_columns(
            (pl.col('periodo').dt.year() * 100 + pl.col('periodo').dt.month())
            .cast(pl.Int64).alias('periodo'))
    if dt == pl.Utf8:
        print("  periodo venia como texto -> se convierte a Int64 AAAAMM")
        return df.with_columns(
            pl.col('periodo').str.replace_all(r'\D', '').str.slice(0, 6)
              .cast(pl.Int64).alias('periodo'))
    return df.with_columns(pl.col('periodo').cast(pl.Int64))


if PARAM['fuente'] == 'preprocesado':
    disp = sorted(DIR_PRE.glob("*.parquet"))
    if not disp:
        raise FileNotFoundError(f"No hay parquet en {DIR_PRE}. Corre 01_Preprocesamiento "
                                f"o usa fuente='crudo'.")
    if PARAM['archivo_preprocesado']:
        path_pre = DIR_PRE / PARAM['archivo_preprocesado']
        if not path_pre.exists():
            raise FileNotFoundError(f"No existe {path_pre}.\nDisponibles: "
                                    f"{[p.name for p in disp]}")
    else:
        path_pre = max(disp, key=lambda p: p.stat().st_size)
    print(f"Leyendo {path_pre.name}")
    print(f"  (disponibles: {[p.name for p in disp]})")

    raw = pl.read_parquet(path_pre)
    faltan = [c for c in ["product_id", "customer_id", "periodo", "tn"] if c not in raw.columns]
    if faltan:
        raise ValueError(f"El parquet no tiene {faltan}. Columnas: {raw.columns}")
    raw = normalizar_periodo(raw)
    cats_ok = [c for c in CATS if c in raw.columns]
    panel = (raw.group_by(["product_id", "customer_id", "periodo"])
                .agg(pl.col("tn").sum().alias("tn"))
                .join(raw.select(["product_id"] + cats_ok).unique(subset=["product_id"]),
                      on="product_id", how="left"))
else:
    sell = normalizar_periodo(
        pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t"))
    prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
              .unique(subset=["product_id"]))
    cats_ok = [c for c in CATS if c in prod.columns]
    panel = (sell.group_by(["product_id", "customer_id", "periodo"])
                 .agg(pl.col("tn").sum().alias("tn"))
                 .join(prod.select(["product_id"] + cats_ok), on="product_id", how="left"))

panel = panel.with_columns(a_m(pl.col("periodo")).alias("m"))

# ── Corte anti-leakage ──────────────────────────────────────────────────────
if PARAM['mes_corte'] is not None:
    m_corte = a_m(PARAM['mes_corte'])
    antes = panel.height
    panel = panel.filter(pl.col("m") < m_corte)
    print(f"\nCorte anti-leakage en {PARAM['mes_corte']}: "
          f"{antes:,} -> {panel.height:,} filas")

M_MIN, M_MAX = int(panel["m"].min()), int(panel["m"].max())
print(f"\npanel: {panel.height:,} filas")
print(f"  {panel['product_id'].n_unique()} productos x {panel['customer_id'].n_unique()} clientes")
print(f"  pares con al menos una venta: {panel.select('product_id','customer_id').n_unique():,}")
print(f"  meses {m_a_periodo(M_MIN)} -> {m_a_periodo(M_MAX)}  ({M_MAX - M_MIN + 1} meses)")
print(f"[{time.time()-t0:.0f}s]")

## 3 — Densificación: ceros desde la primera venta

La regla: **una vez que el par vendió por primera vez, todo mes sin registro es un 0.**

Antes de la primera venta no se rellena nada — ahí la ausencia es real, el par no
existía todavía, y poner ceros inventaría historia. Después de la primera venta la
ausencia significa "no compró", que es información sobre la forma de la serie.

Con `densificar='desde_nacimiento'` la serie llega hasta el final del panel, así que
un par discontinuado arrastra una cola de ceros y eso es exactamente lo que se quiere
que el cluster capture. Con `'vida'` la serie termina en la última venta.

In [ ]:
t0 = time.time()
KEYS = ["product_id", "customer_id"]

vida = (panel.group_by(KEYS)
             .agg(pl.col("m").min().alias("m_nace"),
                  pl.col("m").max().alias("m_ultima"),
                  pl.col("tn").sum().alias("tn_total"),
                  pl.len().alias("meses_con_venta")))

# Hasta donde llega la serie de cada par
if PARAM['densificar'] == 'desde_nacimiento':
    vida = vida.with_columns(pl.lit(M_MAX).alias("m_fin"))
else:
    vida = vida.with_columns(pl.col("m_ultima").alias("m_fin"))

vida = vida.with_columns((pl.col("m_fin") - pl.col("m_nace") + 1).alias("largo"))

# Filtro de largo ANTES de expandir: expandir todo y despues filtrar aloca de mas.
elegibles = vida.filter(pl.col("largo") >= PARAM['min_meses'])
print(f"pares: {vida.height:,} totales -> {elegibles.height:,} con >= "
      f"{PARAM['min_meses']} meses de serie")

# Muestreo de pares para el ajuste final (None = todos). Se toman los de mayor tn:
# los pares chicos son casi todos ruido y aportan poco a la forma de los clusters.
if PARAM['muestra_pares'] and elegibles.height > PARAM['muestra_pares']:
    elegibles = elegibles.sort("tn_total", descending=True).head(PARAM['muestra_pares'])
    print(f"  muestreados a los {PARAM['muestra_pares']:,} de mayor tn")

# ── La grilla densa: una fila por par-mes desde el nacimiento hasta m_fin ────
grilla = (elegibles.select(KEYS + ["m_nace", "m_fin"])
                   .with_columns(pl.int_ranges("m_nace", pl.col("m_fin") + 1).alias("m"))
                   .explode("m")
                   .select(KEYS + ["m"]))

denso = (grilla.join(panel.select(KEYS + ["m", "tn"]), on=KEYS + ["m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0))
               .sort(KEYS + ["m"]))

_ceros = int((denso["tn"] == 0).sum())
print(f"\npanel denso: {denso.height:,} filas   "
      f"({_ceros:,} ceros = {100*_ceros/denso.height:.0f}%)")
print(f"  rellenados con 0: {denso.height - panel.height:,} par-mes que no tenian registro")
print(f"[{time.time()-t0:.0f}s]")

## 4 — Escalado

Cuatro variantes, todas adimensionales salvo `'ninguno'`, que está para mostrar el
problema que las otras resuelven.

| | Fórmula | Media | ¿Qué pasa con los ceros? |
|---|---|---|---|
| `media` | `x / media(x)` | 1 | **siguen en 0** |
| `zscore` | `(x − media) / desvío` | 0 | van a un negativo distinto por serie |
| `maximo` | `x / max(x)` | — | siguen en 0, todo en [0,1] |
| `ninguno` | `x` | — | — |

La razón de preferir `media` en estos datos es la columna de la derecha. Los pares
producto-cliente son muy intermitentes: buena parte de los meses son 0. Con `zscore`
ese 0 se convierte en `−media/desvío`, que vale distinto en cada serie, y dos pares
igual de intermitentes dejan de parecerse entre sí. Con `media` el 0 es 0 en todas.

In [ ]:
def escalar(v: np.ndarray, modo: str) -> np.ndarray:
    """Lleva una serie a escala comparable. Devuelve float64 contiguo, que es lo que
    pide el backend C de dtaidistance."""
    v = np.asarray(v, dtype=np.float64)
    if modo == 'media':
        mu = v.mean()
        out = v / mu if abs(mu) > 1e-9 else v
    elif modo == 'zscore':
        sd = v.std()
        out = (v - v.mean()) / sd if sd > 1e-9 else v - v.mean()
    elif modo == 'maximo':
        mx = np.abs(v).max()
        out = v / mx if mx > 1e-9 else v
    elif modo == 'ninguno':
        out = v
    else:
        raise ValueError(f"escalado invalido: {modo!r}")
    return np.ascontiguousarray(out, dtype=np.float64)


def armar_series(denso_df: pl.DataFrame, modo: str):
    """Devuelve (PARES, SERIES_escaladas, SERIES_crudas).

    PARES es la lista de (product_id, customer_id) en el MISMO orden que las series:
    de ese orden depende poder devolverle la etiqueta de cluster a cada par.
    """
    g = (denso_df.sort(KEYS + ["m"])
                 .group_by(KEYS, maintain_order=True)
                 .agg(pl.col("tn").alias("serie"), pl.col("m").min().alias("m0")))
    pares = list(zip(g["product_id"].to_list(), g["customer_id"].to_list()))
    crudas = [np.asarray(s, dtype=np.float64) for s in g["serie"].to_list()]
    escaladas = [escalar(s, modo) for s in crudas]
    return pares, escaladas, crudas, g["m0"].to_list()


PARES, SERIES, CRUDAS, M0 = armar_series(denso, PARAM['escalado'])
LARGOS = np.array([len(s) for s in SERIES])

print(f"{len(SERIES):,} series   escalado: {PARAM['escalado']}")
print(f"largo  min/mediana/max: {LARGOS.min()} / {int(np.median(LARGOS))} / {LARGOS.max()}")
_todo = np.concatenate(SERIES)
print(f"valores escalados  min/media/max: {_todo.min():.2f} / {_todo.mean():.2f} / {_todo.max():.2f}")
print(f"fraccion de ceros exactos: {100*np.mean(_todo == 0):.0f}%")
del _todo
gc.collect()

### 4.1 — Las series antes y después de escalar

Seis pares de volúmenes muy distintos. **Arriba** en toneladas: la escala del eje la
manda el par más grande y los demás quedan pegados al piso — eso es exactamente lo que
DTW vería, y por lo que agruparía por volumen. **Abajo**, cada uno dividido por su
propia media: la misma forma, ahora comparable.

In [ ]:
# Pares de volumen bien distinto, para que el problema se vea: se ordena por tn_total
# y se toman seis a lo largo de todo el rango, no los seis primeros.
_orden = np.argsort([-c.sum() for c in CRUDAS])
_sel = [_orden[int(f * (len(_orden) - 1))] for f in (0.0, 0.05, 0.2, 0.4, 0.65, 0.9)]

fig, axes = plt.subplots(2, 1, figsize=(10, 6.4))
for ax, datos, tit, ylab in (
        (axes[0], CRUDAS, "Antes de escalar — toneladas. La escala la manda el par mas grande",
         "tn"),
        (axes[1], SERIES, f"Despues de escalar por '{PARAM['escalado']}' — misma forma, comparables",
         "x / media")):
    for j, i in enumerate(_sel):
        v = datos[i]
        eje = np.arange(M0[i], M0[i] + len(v))
        ax.plot(eje, v, color=SERIE[j], linewidth=1.4,
                label=f"{PARES[i][0]}-{PARES[i][1]}  ({CRUDAS[i].sum():,.0f} tn)")
    limpiar(ax, tit, ylab, "mes (indice continuo)")
    ax.legend(ncols=3, fontsize=7.5, loc="upper left")
if PARAM['escalado'] == 'media':
    axes[1].axhline(1.0, color=MUDO, linewidth=.8, linestyle=":")
fig.tight_layout()
guardar(fig, "01_antes_y_despues_de_escalar")

## 5 — k-means con DTW y centroides DBA

Tres piezas:

- **`asignar`** — para cada serie, la distancia DTW a los `k` centroides y el más
  cercano. Es `n·k` distancias, y es todo el costo del algoritmo.
- **`dba`** — el centroide nuevo. Se alinea cada miembro contra el centroide viejo con
  `warping_path` y se promedian los valores que cayeron alineados a cada posición.
  Promediar sin alinear daría la media euclídea y desharía el trabajo de DTW.
- **`kmeans_dtw`** — inicializa con k-means++ (el primer centroide al azar, cada
  siguiente con probabilidad proporcional a la distancia² al más cercano ya elegido),
  y alterna asignar/actualizar hasta que casi nadie cambia de cluster.

Los centroides tienen el largo de la serie que los originó, y las series miembro
pueden tener otro: DTW no necesita que coincidan, y eso es justamente por lo que este
enfoque tolera pares con vidas de distinta duración sin rellenar ni recortar.

In [ ]:
def banda(a, b, window):
    """Banda de Sakoe-Chiba factible para este par de series.

    OJO: esto arruina la corrida entera si no se maneja. La banda impone |i - j| <=
    window, asi que si dos series difieren en MAS de `window` meses de largo no existe
    ningun camino de alineacion valido y dtaidistance devuelve inf. Con
    densificar='desde_nacimiento' los pares nacen en meses distintos y los largos
    difieren mucho: con window=3 la mayoria de las distancias serian inf y el
    clustering agruparia por nada.

    Se ensancha la banda lo justo para que el par sea factible. Donde los largos son
    parecidos la restriccion de desfase se respeta tal cual; donde no, se paga solo la
    diferencia de largo y ni un mes mas.
    """
    if window is None:
        return None
    return max(int(window), abs(len(a) - len(b)))


def d_dtw(a, b, window=None):
    # use_pruning=False a proposito: con series de largos distintos la cota superior que
    # usa el pruning no siempre es valida y dtaidistance devuelve inf en algunos pares
    # (medido: ~1 de cada 500). Cada inf es una asignacion mal hecha. Con series de 20-40
    # meses el pruning casi no acelera, asi que la exactitud sale gratis.
    return dtw.distance_fast(a, b, window=banda(a, b, window), use_pruning=False)


def sanear(D):
    """Reemplaza distancias no finitas por un tope, para que no ganen ningun argmin."""
    mal = ~np.isfinite(D)
    if not mal.any():
        return D, 0
    fin = D[~mal]
    D[mal] = (fin.max() * 10.0) if fin.size else 1.0
    return D, int(mal.sum())


def asignar(series, centros, window=None):
    """Devuelve (labels, dist_al_propio, matriz n x k). n*k distancias DTW."""
    n, k = len(series), len(centros)
    D = np.empty((n, k), dtype=np.float64)
    for j, c in enumerate(centros):
        cj = np.ascontiguousarray(c, dtype=np.float64)
        for i, s in enumerate(series):
            D[i, j] = d_dtw(s, cj, window)
    D, n_mal = sanear(D)
    if n_mal:
        print(f"   aviso: {n_mal:,} de {n*k:,} distancias no finitas (banda infactible)")
    lab = D.argmin(axis=1)
    return lab, D[np.arange(n), lab], D


def dba(miembros, centro, window=None, iters=1):
    """DTW Barycenter Averaging: promedia los puntos ALINEADOS contra el centroide."""
    centro = np.ascontiguousarray(centro, dtype=np.float64)
    if not miembros:
        return centro
    T = len(centro)
    for _ in range(iters):
        acum = np.zeros(T, dtype=np.float64)
        cuenta = np.zeros(T, dtype=np.float64)
        for s in miembros:
            w = banda(centro, s, window)
            path = (dtw.warping_path(centro, s, window=w) if _WP_WINDOW
                    else dtw.warping_path(centro, s))
            for i, j in path:
                acum[i] += s[j]
                cuenta[i] += 1.0
        centro = np.where(cuenta > 0, acum / np.maximum(cuenta, 1.0), centro)
        centro = np.ascontiguousarray(centro, dtype=np.float64)
    return centro


def init_kmeanspp(series, k, window, rng):
    """k-means++ con distancia DTW. Cuesta k*n distancias, lo mismo que una asignacion."""
    n = len(series)
    centros = [series[int(rng.integers(n))].copy()]
    d_min = np.array([d_dtw(s, centros[0], window) for s in series])
    for _ in range(1, k):
        p = d_min ** 2
        tot = p.sum()
        idx = int(rng.integers(n)) if tot <= 0 else int(rng.choice(n, p=p / tot))
        centros.append(series[idx].copy())
        d_nuevo = np.array([d_dtw(s, centros[-1], window) for s in series])
        d_min = np.minimum(d_min, d_nuevo)
    return centros


def kmeans_dtw(series, k, window=None, max_iter=15, tol=0.01, dba_iters=1,
               semilla=0, verbose=True):
    """k-means con DTW y centroides DBA. Memoria O(n*k), no O(n^2)."""
    rng = np.random.default_rng(semilla)
    n = len(series)
    centros = init_kmeanspp(series, k, window, rng)
    lab = np.full(n, -1)
    hist = []

    for it in range(1, max_iter + 1):
        lab_new, d_prop, _ = asignar(series, centros, window)
        cambios = int((lab_new != lab).sum())
        lab = lab_new
        inercia = float(d_prop.sum())
        hist.append({'iter': it, 'inercia': inercia, 'cambios': cambios,
                     'frac_cambios': cambios / n})
        if verbose:
            tam = np.bincount(lab, minlength=k)
            print(f"  iter {it:2d}  inercia {inercia:12,.1f}  "
                  f"cambian {cambios:6,} ({100*cambios/n:5.2f}%)  tam {tam.tolist()}")

        # Cluster vacio: se siembra con la serie mas lejana a su propio centroide, que
        # es la peor explicada por la particion actual. Sin esto k se degrada solo.
        for j in range(k):
            if not np.any(lab == j):
                peor = int(np.argmax(d_prop))
                centros[j] = series[peor].copy()
                lab[peor] = j

        centros = [dba([series[i] for i in np.flatnonzero(lab == j)], centros[j],
                       window, dba_iters)
                   for j in range(k)]

        if cambios / n <= tol and it > 1:
            if verbose:
                print(f"  convergio: cambia {100*cambios/n:.2f}% <= {100*tol:.2f}%")
            break

    lab, d_prop, D = asignar(series, centros, window)
    return {'labels': lab, 'centros': centros, 'inercia': float(d_prop.sum()),
            'dist_propio': d_prop, 'D': D, 'hist': hist, 'k': k, 'window': window}

### 5.1 — Silhouette sobre una muestra

El silhouette necesita las distancias de todos contra todos, que es la matriz que
estamos evitando. Se estima sobre una muestra: se eligen `muestra_silhouette` series
al azar, se calcula su matriz DTW completa —chica y manejable— y se evalúa el
silhouette de sus etiquetas ahí. Es una estimación; siempre se reporta con cuántas
series se hizo.

La muestra se toma **estratificada por cluster**, para que no se caiga un cluster
chico y el número se calcule sobre menos grupos de los que hay.

In [ ]:
def matriz_dtw(series, window=None):
    """Matriz simetrica completa. Solo para muestras chicas: es O(n^2)."""
    n = len(series)
    D = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        si = series[i]
        for j in range(i + 1, n):
            D[i, j] = D[j, i] = d_dtw(si, series[j], window)
    D, _ = sanear(D)
    return D


def silhouette_muestra(series, labels, window, n_muestra, rng):
    """Silhouette estimado sobre una muestra estratificada por cluster."""
    labels = np.asarray(labels)
    n = len(series)
    if n <= n_muestra:
        idx = np.arange(n)
    else:
        # Estratificado: de cada cluster, una cuota proporcional y al menos 2.
        idx = []
        for j in np.unique(labels):
            en_j = np.flatnonzero(labels == j)
            cuota = max(2, int(round(n_muestra * len(en_j) / n)))
            cuota = min(cuota, len(en_j))
            idx.append(rng.choice(en_j, size=cuota, replace=False))
        idx = np.concatenate(idx)
    lab_m = labels[idx]
    if len(np.unique(lab_m)) < 2:
        return float('nan'), len(idx)
    D = matriz_dtw([series[i] for i in idx], window)
    return float(silhouette_score(D, lab_m, metric='precomputed')), len(idx)


def resumen_particion(labels, dist_propio, series):
    """Metricas que no necesitan la matriz n x n: tamanios, balance, inercia."""
    labels = np.asarray(labels)
    tam = np.bincount(labels, minlength=labels.max() + 1)
    return {
        'k_efectivo': int((tam > 0).sum()),
        'tam_min': int(tam.min()), 'tam_max': int(tam.max()),
        'frac_min': float(tam.min() / len(labels)),
        'inercia': float(np.sum(dist_propio)),
        'inercia_media': float(np.mean(dist_propio)),
    }

## 6 — Grilla de parámetros del DTW

Se cruzan `window` × `escalado` × `k` sobre una muestra de pares
(`muestra_grilla`), porque la grilla ajusta un k-means completo por combinación. El
objetivo es **ordenar** configuraciones, no producir la partición final.

Tres columnas para leerla, y ninguna alcanza sola:

- **`silhouette`** — separación de los clusters, estimada sobre muestra. Sube
  artificialmente cuando la partición desprende un puñado de outliers, así que hay que
  mirarla junto con la siguiente.
- **`frac_min`** — el cluster más chico como fracción del total. Si es 0,001 el
  silhouette alto no significa nada: son outliers separados del resto.
- **`inercia_media`** — distancia DTW promedio de cada serie a su centroide. Baja
  siempre que sube k, así que sirve para comparar entre configuraciones **con el mismo
  k**, no entre k distintos.

`escalado` cambia la escala de las distancias, así que la inercia **no** es comparable
entre escalados. El silhouette sí, porque es adimensional.

In [ ]:
# Muestra para la grilla: los pares de mayor tn de la muestra ya filtrada.
_n_g = min(PARAM['muestra_grilla'], len(SERIES))
_ord_tn = np.argsort([-c.sum() for c in CRUDAS])[:_n_g]
IDX_G = np.sort(_ord_tn)
denso_g = denso.join(
    pl.DataFrame({'product_id': [PARES[i][0] for i in IDX_G],
                  'customer_id': [PARES[i][1] for i in IDX_G]}),
    on=KEYS, how='inner')
print(f"grilla sobre {len(IDX_G):,} pares (de {len(SERIES):,})")

filas = []
t0 = time.time()
for esc in PARAM['grilla_escalado']:
    _, S_esc, _, _ = armar_series(denso_g, esc)
    for win in PARAM['grilla_window']:
        for k in PARAM['grilla_k']:
            r = kmeans_dtw(S_esc, k, window=win, max_iter=PARAM['max_iter'],
                           tol=PARAM['tol_cambio'], dba_iters=PARAM['dba_iters'],
                           semilla=PARAM['semilla'], verbose=False)
            sil, n_sil = silhouette_muestra(S_esc, r['labels'], win,
                                            PARAM['muestra_silhouette'],
                                            np.random.default_rng(PARAM['semilla']))
            res = resumen_particion(r['labels'], r['dist_propio'], S_esc)
            filas.append({'escalado': esc, 'window': (win if win is not None else -1),
                          'k': k, 'silhouette': round(sil, 4),
                          'n_silhouette': n_sil, **{a: round(b, 4) if isinstance(b, float)
                                                    else b for a, b in res.items()},
                          'iters': len(r['hist'])})
            print(f"  {esc:7s} w={str(win):4s} k={k:2d} -> sil {sil:+.4f}  "
                  f"frac_min {res['frac_min']:.3f}  inercia_media "
                  f"{res['inercia_media']:8.3f}   [{time.time()-t0:.0f}s]", flush=True)
            del r
            gc.collect()

grid = pl.DataFrame(filas)
grid.write_csv(DIR_RUN / "grilla_dtw.csv")
print(f"\nGuardado: {(DIR_RUN / 'grilla_dtw.csv').relative_to(BUCKET)}")

# window=-1 en el CSV es None (sin banda): -1 para que la columna quede numerica.
print("\nMejores por silhouette, entre las particiones balanceadas (frac_min >= 0.02):")
print(grid.filter(pl.col("frac_min") >= 0.02).sort("silhouette", descending=True).head(12))

In [ ]:
# Silhouette vs k, una linea por window, un panel por escalado.
escs = PARAM['grilla_escalado']
fig, axes = plt.subplots(1, len(escs), figsize=(4.2 * len(escs), 3.6), sharey=True)
axes = np.atleast_1d(axes)
for ax, esc in zip(axes, escs):
    g = grid.filter(pl.col("escalado") == esc)
    for j, win in enumerate(PARAM['grilla_window']):
        w = win if win is not None else -1
        gg = g.filter(pl.col("window") == w).sort("k")
        if gg.height == 0:
            continue
        ax.plot(gg["k"], gg["silhouette"], color=SERIE[j], linewidth=1.8,
                marker="o", markersize=4, label=f"w={win}")
        # marca hueca donde la particion NO esta balanceada
        mal = gg.filter(pl.col("frac_min") < 0.02)
        if mal.height:
            ax.scatter(mal["k"], mal["silhouette"], s=55, facecolors=FONDO,
                       edgecolors=SERIE[j], linewidths=1.4, zorder=3)
    limpiar(ax, f"escalado = {esc}", "silhouette (muestra)", "k")
axes[0].legend(fontsize=7.5, loc="best")
fig.suptitle("Marca hueca = el cluster mas chico es menos del 2% -> silhouette no confiable",
             color=TINTA2, fontsize=8.5, y=1.02)
fig.tight_layout()
guardar(fig, "02_grilla_silhouette")

## 7 — Ajuste final

Con las palancas de `PARAM` (`escalado`, `window`, `k`), ahora sobre **todos** los
pares que pasaron el filtro, no sobre la muestra de la grilla. Si la grilla sugirió
otra combinación, se cambia arriba y se vuelve a correr desde la sección 4: los
resultados de la grilla son para decidir, no se aplican solos.

In [ ]:
t0 = time.time()
print(f"k-means DTW: {len(SERIES):,} series, k={PARAM['k']}, window={PARAM['window']}, "
      f"escalado={PARAM['escalado']}")
print(f"  costo por iteracion: {len(SERIES) * PARAM['k']:,} distancias DTW")

RES = kmeans_dtw(SERIES, PARAM['k'], window=PARAM['window'],
                 max_iter=PARAM['max_iter'], tol=PARAM['tol_cambio'],
                 dba_iters=PARAM['dba_iters'], semilla=PARAM['semilla'])
LAB = RES['labels']
K = PARAM['k']
print(f"[{time.time()-t0:.0f}s]")

SIL, N_SIL = silhouette_muestra(SERIES, LAB, PARAM['window'],
                                PARAM['muestra_silhouette'], RNG)
RESUMEN = resumen_particion(LAB, RES['dist_propio'], SERIES)
print(f"\nsilhouette (sobre {N_SIL:,} series): {SIL:+.4f}")
print(f"resumen: {RESUMEN}")

# La tabla de clusters con lo que hace falta para leerlos
tn_par = np.array([c.sum() for c in CRUDAS])
meses_con_venta = np.array([int((c > 0).sum()) for c in CRUDAS])

etiquetas = pl.DataFrame({
    'product_id':  [p for p, _ in PARES],
    'customer_id': [c for _, c in PARES],
    'cluster':     LAB.astype(np.int32),
    'dist_centroide': RES['dist_propio'],
    'tn_total':    tn_par,
    'largo':       LARGOS,
    'meses_con_venta': meses_con_venta,
}).with_columns(
    (pl.col("meses_con_venta") / pl.col("largo")).alias("frac_meses_con_venta"))

resumen_cl = (etiquetas.group_by("cluster")
    .agg(pl.len().alias("n_pares"),
         pl.col("product_id").n_unique().alias("n_productos"),
         pl.col("customer_id").n_unique().alias("n_clientes"),
         pl.col("tn_total").sum().alias("tn"),
         pl.col("tn_total").median().alias("tn_mediana_par"),
         pl.col("largo").median().alias("largo_mediano"),
         pl.col("frac_meses_con_venta").mean().alias("frac_meses_con_venta"),
         pl.col("dist_centroide").mean().alias("dist_media"))
    .with_columns((100 * pl.col("n_pares") / etiquetas.height).round(1).alias("pct_pares"),
                  (100 * pl.col("tn") / etiquetas["tn_total"].sum()).round(1).alias("pct_tn"))
    .sort("n_pares", descending=True))

print()
print(resumen_cl)
print("\nfrac_meses_con_venta: 1.0 = vendio todos los meses de su vida; "
      "0.2 = solo en 1 de cada 5.")

## 8 — Los clusters, caracterizados

### 8.1 — Forma: centroide y miembros

Un panel por cluster. En gris, una muestra de las series miembro; en color, el
centroide DBA; la banda es el rango p10–p90 mes a mes. El centroide es la forma que
el cluster representa, y la banda dice cuánto se le parecen realmente sus miembros: un
cluster con banda ancha está agrupando cosas que no se parecen tanto.

Las series se alinean por **posición dentro de su propia serie** (mes 0 = primer mes de
cada par), no por calendario: es lo que hace comparables pares que nacieron en meses
distintos.

In [ ]:
N_MUESTRA_PLOT = 60
orden_cl = resumen_cl["cluster"].to_list()
ncol = min(3, K)
nfil = int(np.ceil(K / ncol))
fig, axes = plt.subplots(nfil, ncol, figsize=(4.6 * ncol, 3.0 * nfil), squeeze=False)

for pos, cl in enumerate(orden_cl):
    ax = axes[pos // ncol][pos % ncol]
    miembros = np.flatnonzero(LAB == cl)
    cen = RES['centros'][cl]
    T = max(len(cen), int(np.median([len(SERIES[i]) for i in miembros])))

    # matriz miembros x T, con nan donde la serie es mas corta -> percentiles por mes
    sub = miembros if len(miembros) <= N_MUESTRA_PLOT else RNG.choice(
        miembros, N_MUESTRA_PLOT, replace=False)
    M = np.full((len(sub), T), np.nan)
    for r, i in enumerate(sub):
        v = SERIES[i][:T]
        M[r, :len(v)] = v
        ax.plot(np.arange(len(v)), v, color=GRILLA, linewidth=.7, zorder=1)

    with np.errstate(all='ignore'):
        p10 = np.nanpercentile(M, 10, axis=0)
        p90 = np.nanpercentile(M, 90, axis=0)
    ax.fill_between(np.arange(T), p10, p90, color=SERIE[pos % len(SERIE)],
                    alpha=.18, zorder=2, linewidth=0)
    ax.plot(np.arange(len(cen)), cen, color=SERIE[pos % len(SERIE)], linewidth=2.2,
            zorder=3)

    fila = resumen_cl.filter(pl.col("cluster") == cl).to_dicts()[0]
    limpiar(ax, f"cluster {cl} — {fila['n_pares']:,} pares ({fila['pct_pares']}%), "
                f"{fila['pct_tn']}% de tn\n"
                f"vende en {100*fila['frac_meses_con_venta']:.0f}% de sus meses",
            f"x / {PARAM['escalado']}", "mes desde el inicio del par")

for pos in range(K, nfil * ncol):
    axes[pos // ncol][pos % ncol].axis("off")
fig.tight_layout()
guardar(fig, f"03_forma_por_cluster_k{K}")

### 8.2 — Composición por categorías

La pregunta de fondo: **¿los clusters de forma dicen algo que la jerarquía comercial
no diga ya?** Si cada cluster fuera casi una `cat3`, el clustering sería una forma
caravanera de recuperar información que ya está en el dataset.

Se mide con **lift**: la participación de una categoría dentro del cluster dividida
por su participación global. Lift 3 = esa categoría está tres veces más representada
acá que en el total. Lift ≈ 1 en todo el panel = el cluster no tiene sesgo de
categoría, y su información es genuinamente nueva.

In [ ]:
cats_disp = [c for c in CATS if c in panel.columns]
pc_cat = panel.select(KEYS + cats_disp).unique(subset=KEYS)
et_cat = etiquetas.join(pc_cat, on=KEYS, how="left")

TABLAS = {}
for c in cats_disp:
    glob = (et_cat.group_by(c).agg(pl.len().alias("n_glob"))
                  .with_columns((pl.col("n_glob") / et_cat.height).alias("sh_glob")))
    porcl = (et_cat.group_by(["cluster", c]).agg(pl.len().alias("n"))
                   .join(et_cat.group_by("cluster").agg(pl.len().alias("n_cl")),
                         on="cluster")
                   .with_columns((pl.col("n") / pl.col("n_cl")).alias("sh_cl"))
                   .join(glob, on=c, how="left")
                   .with_columns((pl.col("sh_cl") / pl.col("sh_glob")).alias("lift"))
                   .sort(["cluster", "lift"], descending=[False, True]))
    TABLAS[c] = porcl
    porcl.write_csv(DIR_RUN / f"composicion_{c}_k{K}.csv")

    print(f"\n{'='*70}\n{c.upper()}: lo mas sobrerrepresentado de cada cluster "
          f"(min 20 pares)\n{'='*70}")
    top = (porcl.filter(pl.col("n") >= 20)
                .group_by("cluster", maintain_order=True).head(3)
                .select("cluster", c, "n", "sh_cl", "sh_glob", "lift")
                .with_columns(pl.col("sh_cl").round(3), pl.col("sh_glob").round(3),
                              pl.col("lift").round(2)))
    print(top)

# Concentracion: cuanto del cluster se lo lleva su categoria dominante.
print(f"\n{'='*70}\nCONCENTRACION por cluster: peso de su cat3 dominante\n{'='*70}")
if "cat3" in TABLAS:
    dom = (TABLAS["cat3"].sort(["cluster", "sh_cl"], descending=[False, True])
                         .group_by("cluster", maintain_order=True).head(1)
                         .select("cluster", pl.col("cat3").alias("cat3_dominante"),
                                 pl.col("sh_cl").round(3).alias("peso_en_cluster"),
                                 pl.col("lift").round(2)))
    print(dom)
    print("\npeso_en_cluster alto (>0.5) = el cluster es casi una sola cat3, y entonces "
          "aporta poco sobre la jerarquia.\nRepartido = agrupa por forma, cruzando "
          "categorias: eso es informacion nueva.")

In [ ]:
# Heatmap cluster x cat3 (lift, log2). Solo las cat3 con presencia suficiente para que
# el lift no sea ruido de conteos chicos.
if "cat3" in TABLAS:
    top_cats = (et_cat.group_by("cat3").agg(pl.len().alias("n"))
                      .sort("n", descending=True).head(14)["cat3"].to_list())
    M = np.ones((K, len(top_cats)))
    t = TABLAS["cat3"]
    mapa = {(r["cluster"], r["cat3"]): r["lift"] for r in t.to_dicts()}
    for i, cl in enumerate(orden_cl):
        for j, ca in enumerate(top_cats):
            M[i, j] = mapa.get((cl, ca), np.nan)

    L = np.log2(np.where(np.isfinite(M) & (M > 0), M, np.nan))
    vmax = float(np.nanmax(np.abs(L))) if np.isfinite(L).any() else 1.0

    fig, ax = plt.subplots(figsize=(1 + .62 * len(top_cats), 1.2 + .46 * K))
    im = ax.imshow(L, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(top_cats)))
    ax.set_xticklabels([str(c)[:14] for c in top_cats], rotation=45, ha="right", fontsize=7.5)
    ax.set_yticks(range(K))
    ax.set_yticklabels([f"cl {c}" for c in orden_cl], fontsize=8)
    for i in range(K):
        for j in range(len(top_cats)):
            if np.isfinite(M[i, j]):
                ax.text(j, i, f"{M[i,j]:.1f}", ha="center", va="center", fontsize=6.5,
                        color=TINTA if abs(L[i, j]) < vmax * .55 else FONDO)
    ax.grid(False)
    ax.set_title("Lift de cada cat3 por cluster — rojo: sobrerrepresentada, azul: sub\n"
                 "1.0 en todo el panel = el cluster no tiene sesgo de categoria",
                 color=TINTA, loc="left", pad=10, fontsize=9)
    fig.colorbar(im, ax=ax, label="log2(lift)", fraction=.025)
    fig.tight_layout()
    guardar(fig, f"04_lift_cat3_k{K}")

### 8.3 — Series concretas de un cluster, sin escalar y escaladas

Para cerrar el círculo del principio: los mismos pares de un cluster, primero en
toneladas y después escalados. Es la prueba de por qué quedaron juntos — en
toneladas pueden verse muy distintos, y es la forma escalada la que los agrupó.

Cambiá `CL_VER` para recorrer los clusters.

In [ ]:
CL_VER = int(orden_cl[0])       # cambiar para mirar otro cluster

miembros = np.flatnonzero(LAB == CL_VER)
# Los mas representativos: los mas cercanos al centroide.
mas_cerca = miembros[np.argsort(RES['dist_propio'][miembros])[:6]]

fig, axes = plt.subplots(2, 1, figsize=(10, 6.4))
for j, i in enumerate(mas_cerca):
    ax = axes[0]
    ax.plot(np.arange(len(CRUDAS[i])), CRUDAS[i], color=SERIE[j], linewidth=1.4,
            label=f"{PARES[i][0]}-{PARES[i][1]}  ({CRUDAS[i].sum():,.0f} tn)")
    axes[1].plot(np.arange(len(SERIES[i])), SERIES[i], color=SERIE[j], linewidth=1.4)

cen = RES['centros'][CL_VER]
axes[1].plot(np.arange(len(cen)), cen, color=TINTA, linewidth=2.4, linestyle="--",
             label="centroide DBA")
limpiar(axes[0], f"cluster {CL_VER} — los 6 mas cercanos al centroide, en toneladas",
        "tn", "mes desde el inicio del par")
limpiar(axes[1], f"los mismos, escalados por '{PARAM['escalado']}' — asi los vio el DTW",
        f"x / {PARAM['escalado']}", "mes desde el inicio del par")
axes[0].legend(ncols=3, fontsize=7.5)
axes[1].legend(fontsize=8)
fig.tight_layout()
guardar(fig, f"05_cluster{CL_VER}_crudo_vs_escalado_k{K}")

## 9 — Guardar

Dos archivos que sirven después:

- **`clusters_pc_*.parquet`** en `datasets_fe/` — una fila por par con su etiqueta.
  Se joinea por `(product_id, customer_id)` en `02_FE` como feature categórica.
- **`resultado.json`** en la carpeta del experimento — todas las palancas y las
  métricas, para poder comparar corridas y saber cuál generó qué parquet.

Los pares que no entraron al clustering (menos de `min_meses`) **no** están en el
parquet: al joinear quedan nulos, y ese nulo es informativo — significa "par con
historia demasiado corta para tener forma". Conviene dejarlo como categoría propia en
`02_FE` y no rellenarlo con un cluster cualquiera.

In [ ]:
NOMBRE = (f"clusters_pc_{PARAM['densificar']}_{PARAM['escalado']}"
          f"_w{PARAM['window']}_k{K}_min{PARAM['min_meses']}_corte{PARAM['mes_corte']}")

salida = etiquetas.select("product_id", "customer_id",
                          pl.col("cluster").alias(f"cluster_pc_k{K}"),
                          "dist_centroide")
path_out = DIR_OUT / f"{NOMBRE}.parquet"
salida.write_parquet(path_out)
resumen_cl.write_csv(DIR_RUN / f"resumen_clusters_k{K}.csv")

resultado = {
    'notebook': 'dtw_nuevo',
    'idea': ('clustering de pares producto-cliente por forma de serie, con k-means DTW '
             'y centroides DBA: memoria O(n*k) en vez de la matriz n^2 del jerarquico'),
    'fuente': PARAM['fuente'],
    'archivo_fuente': (str(path_pre.name) if PARAM['fuente'] == 'preprocesado' else 'sell-in.txt.gz'),
    'densificar': PARAM['densificar'],
    'escalado': PARAM['escalado'],
    'window': PARAM['window'],
    'k': K,
    'mes_corte': PARAM['mes_corte'],
    'min_meses': PARAM['min_meses'],
    'n_pares_clusterizados': int(etiquetas.height),
    'n_pares_totales': int(vida.height),
    'silhouette_muestra': SIL,
    'n_series_silhouette': int(N_SIL),
    **{f'part_{k_}': v for k_, v in RESUMEN.items()},
    'iteraciones': len(RES['hist']),
    'historia': RES['hist'],
    'clusters': resumen_cl.to_dicts(),
    'archivo_salida': str(path_out),
    'semilla': PARAM['semilla'],
}
with open(DIR_RUN / f"resultado_k{K}.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)

print(f"Etiquetas : {path_out}")
print(f"  {salida.height:,} pares de {vida.height:,} "
      f"({100*salida.height/vida.height:.0f}%); el resto queda nulo al joinear")
print(f"\nArchivos en {DIR_RUN.relative_to(BUCKET)}:")
for p in sorted(DIR_RUN.iterdir()):
    print(f"  - {p.name}")
print(f"\nPara usarlo en 02_FE:")
print(f"  cl = pl.read_parquet(r'{path_out}')")
print(f"  df = df.join(cl.drop('dist_centroide'), on=['product_id','customer_id'], how='left')")

## 10 — Qué mirar de todo esto

Tres preguntas concretas, en orden:

1. **¿Los clusters cruzan categorías?** Si en la sección 8.2 el `peso_en_cluster` de
   la `cat3` dominante es bajo y los lifts están cerca de 1, el clustering agrupó por
   comportamiento y no por qué es el producto — eso es información que el modelo hoy
   no tiene. Si un cluster es casi una sola `cat3`, esa parte es redundante con una
   feature que ya existe.

2. **¿Los clusters se distinguen por intermitencia?** La columna
   `frac_meses_con_venta` del resumen suele ser la que más separa los grupos en datos
   como estos: pares que compran todos los meses contra pares esporádicos. Si es lo
   único que los separa, quizá alcance con esa variable sola como feature y el DTW
   sea un camino largo hacia algo simple. Vale la pena saberlo.

3. **¿La banda p10–p90 de la sección 8.1 es angosta?** Si es ancha, el centroide no
   representa a sus miembros y ese cluster no es un arquetipo, es un cajón. Ahí conviene
   subir `k` o revisar el escalado antes de usar las etiquetas como feature.

Y una advertencia sobre el paso siguiente: estas etiquetas se calcularon con datos
**anteriores a `mes_corte`**, pero se aplican a todas las filas del panel, incluidas
las de validación y test. Eso está bien —la etiqueta no vio el futuro— pero un par que
nació después del corte no tiene cluster. Ese nulo hay que tratarlo explícitamente en
`02_FE`, no rellenarlo con el cluster más común.